# 01. EDA - Разведывательный анализ данных

# Постановка задачи

Нам дано три файла:
- train.parquet       - обучающая выборка: пары (запрос, объявление), которые пользователь выбрал
- benchmark_queries.parquet  - 2 452 запроса, для которых нужно найти кандидатов
- benchmark_items.parquet    - 189 212 объявлений, среди которых искать

Задача: для каждого benchmark-запроса найти до 50 объявлений-кандидатов.
Метрика: Recall@50 = mean по запросам ( |топ-50 ∩ релевантные| / |релевантные| )

Цель этого ноутбука:
1. Понять структуру данных
2. Выявить ключевые закономерности, которые определят архитектуру решения
3. Сформулировать гипотезы для каждого компонента пайплайна

## 0. Импорты

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # для запуска без дисплея; уберите строку если работаете в Jupyter
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path

In [ ]:
import os
import requests
from pathlib import Path

# 1. Ссылка на Яндекс.Диск из задания
PUBLIC_URL = "https://disk.yandex.ru/d/sNhfo0YOjGtufg"

print("Получаем прямую ссылку для скачивания через API Яндекс.Диска...")
api_url = f"https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key={PUBLIC_URL}"
response = requests.get(api_url)

if response.status_code == 200:
    download_url = response.json()["href"]
    print("Прямая ссылка получена!")
else:
    raise RuntimeError(f"Ошибка API Яндекс.Диска: {response.status_code}")

# 2. Скачивание архива через wget
archive_name = "avito_data.zip"
print(f"Скачиваем архив (~100-200 МБ)...")
!wget -q --show-progress -O {archive_name} "{download_url}"

Получаем прямую ссылку для скачивания через API Яндекс.Диска...
Прямая ссылка получена!
Скачиваем архив (~100-200 МБ)...
avito_data.zip          [<=>                 ] 651.79M  23.8MB/s    in 28s     
Распаковываем в /content/data...
Проверяем структуру файлов...
Файлы .parquet не найдены! Проверьте содержимое архива:
total 12
drwxr-xr-x 3 root root 4096 Sep 20 01:28 .
drwxr-xr-x 1 root root 4096 Sep 20 01:28 ..
drwxr-xr-x 2 root root 4096 Sep 20 01:28 NLP_avito_interns


In [ ]:
# 3. Распаковка
DATA_DIR = Path("/content/first")
DATA_DIR.mkdir(exist_ok=True)
print(f"Распаковываем в {DATA_DIR}...")
!unzip -q -o {archive_name} -d {DATA_DIR}

Распаковываем в /content/first...


In [ ]:
archive_name = "/content/first/NLP_avito_interns/dataset.zip"
DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(exist_ok=True)
print(f"Распаковываем в {DATA_DIR}...")
!unzip -q -o {archive_name} -d {DATA_DIR}

# 4. Очистка и проверка структуры (Яндекс.Диск иногда создает вложенную папку с именем ссылки)
print("Проверяем структуру файлов...")
parquet_files = list(DATA_DIR.rglob("*.parquet"))

if not parquet_files:
    print("Файлы .parquet не найдены! Проверьте содержимое архива:")
    !ls -la {DATA_DIR}
else:
    # Если файлы лежат во вложенной папке, перемещаем их на уровень выше
    for p in parquet_files:
        target = DATA_DIR / p.name
        if not target.exists():
            p.rename(target)

    # Удаляем пустые вложенные папки и сам архив
    !rm -f {archive_name}
    !rm -rf {DATA_DIR}/sNhfo0YOjGtufg  # возможная вложенная папка

    print("\nДанные успешно загружены и распакованы!")
    !ls -lh {DATA_DIR}/*.parquet

Распаковываем в /content/data...
Проверяем структуру файлов...

Данные успешно загружены и распакованы!
-rw-r--r-- 1 root root 187M Sep 11 11:17 /content/data/benchmark_items.parquet
-rw-r--r-- 1 root root 125K Sep 11 11:17 /content/data/benchmark_queries.parquet
-rw-r--r-- 1 root root 468M Sep 11 11:19 /content/data/train.parquet


## 1. Загрузка и первый взгляд

In [ ]:
train             = pd.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
benchmark_queries = pd.read_parquet(os.path.join(DATA_DIR, "benchmark_queries.parquet"))
benchmark_items   = pd.read_parquet(os.path.join(DATA_DIR, "benchmark_items.parquet"))

# ID - строки. Parquet иногда хранит hex-строки как object, но перестрахуемся
benchmark_items["item_id"]    = benchmark_items["item_id"].astype(str)
benchmark_queries["query_id"] = benchmark_queries["query_id"].astype(str)
train["item_id"]              = train["item_id"].astype(str)

print("=== РАЗМЕРЫ ===")
print(f"  train:             {len(train):>10,} строк,  {len(train.columns)} колонок")
print(f"  benchmark_queries: {len(benchmark_queries):>10,} строк,  {len(benchmark_queries.columns)} колонок")
print(f"  benchmark_items:   {len(benchmark_items):>10,} строк,  {len(benchmark_items.columns)} колонок")

print("\n=== СХЕМА TRAIN ===")
print(train.dtypes.to_string())

print("\n=== СХЕМА BENCHMARK_QUERIES ===")
print(benchmark_queries.dtypes.to_string())

print("\n=== СХЕМА BENCHMARK_ITEMS ===")
print(benchmark_items.dtypes.to_string())

=== РАЗМЕРЫ ===
  train:                497,673 строк,  19 колонок
  benchmark_queries:      2,452 строк,  6 колонок
  benchmark_items:      189,212 строк,  14 колонок

=== СХЕМА TRAIN ===
search_query                  object
search_location_id             int64
search_is_delivery_search      int32
search_infm_params_text       object
search_category                int64
item_title_raw                object
item_rating_reviews_count    float64
item_rating                  float64
item_price                    object
item_microcat_id               int64
item_longitude                object
item_location_id               int64
item_latitude                 object
item_is_phone_hidden            bool
item_is_message_forbidden       bool
item_infm_params_text         object
item_id                       object
item_description_raw          object
item_category_id               int64

=== СХЕМА BENCHMARK_QUERIES ===
query_id                     object
search_query                 object
sea

In [ ]:
print("=== ПРИМЕРЫ ===\n")
print("train (первые 3 строки):")
display_cols = [c for c in ["search_query", "search_category", "item_title_raw",
                             "item_category_id", "search_location_id", "item_location_id"]
                if c in train.columns]
print(train[display_cols].head(3).to_string(index=False))

print("\nbenchmark_queries (первые 3):")
print(benchmark_queries.head(3).to_string(index=False))

print("\nbenchmark_items (первые 3):")
item_cols = [c for c in ["item_id", "item_title_raw", "item_category_id",
                          "item_rating", "item_location_id"] if c in benchmark_items.columns]
print(benchmark_items[item_cols].head(3).to_string(index=False))

=== ПРИМЕРЫ ===

train (первые 3 строки):
      search_query  search_category                        item_title_raw  item_category_id  search_location_id  item_location_id
скупка телевизоров              114                    Скупка б/у техники               114              652430            652430
        автоподбор              114  Автоподбор Разовый осмотр автомобиля               114              640860            640860
    баня на дровах              114 Баня на дровах "Прованс" на Цветочной               114              653240            653240

benchmark_queries (первые 3):
        query_id                  search_query  search_location_id  search_is_delivery_search      search_infm_params_text  search_category
70DfDUpwjxB4lzFd перевозки владикавказ тбилиси              649820                          0                                           114
JTrdTaZJvSiLPkXj                обзвон по базе              107620                          0    Вид услуги Деловые услуги     

## 2. Качество данных: пропуски

In [ ]:
print("=== ПРОПУСКИ В TRAIN ===")
null_train = (train.isnull().mean() * 100).round(1)
print(null_train[null_train > 0].sort_values(ascending=False).to_string())

print("\n=== ПРОПУСКИ В BENCHMARK_QUERIES ===")
null_q = (benchmark_queries.isnull().mean() * 100).round(1)
nz_q = null_q[null_q > 0]
print(nz_q.to_string() if len(nz_q) else "  Пропусков нет")

print("\n=== ПРОПУСКИ В BENCHMARK_ITEMS ===")
null_i = (benchmark_items.isnull().mean() * 100).round(1)
nz_i = null_i[null_i > 0].sort_values(ascending=False)
print(nz_i.to_string() if len(nz_i) else "  Пропусков нет")

=== ПРОПУСКИ В TRAIN ===
item_rating                  5.7
item_rating_reviews_count    3.9

=== ПРОПУСКИ В BENCHMARK_QUERIES ===
  Пропусков нет

=== ПРОПУСКИ В BENCHMARK_ITEMS ===
item_rating                  9.3
item_rating_reviews_count    6.1


### Вывод по пропускам

- Наличие пропусков в item_description_raw и item_infm_params_text нормально:
  не все продавцы заполняют описание и параметры.
- При построении BM25 такие поля просто не добавляются в текст документа.
- Для признаков рейтинга и локации пропуски = "нет данных"; заполняем нулем/мотивом.

## 3. Категории - ключевой структурный признак

**Гипотеза**: пользователь ищет услугу в определенной категории,
и релевантные объявления почти всегда принадлежат той же категории.
Если это подтвердится - категория является обязательным фильтром, а не просто признаком.

In [ ]:
# Распределение объявлений по категориям
if "item_category_id" in benchmark_items.columns:
    cat_item = benchmark_items["item_category_id"].value_counts()
    print(f"Уникальных категорий в benchmark_items: {len(cat_item)}")
    print(f"Объявлений: min={cat_item.min()}, mean={cat_item.mean():.0f}, max={cat_item.max()}")
    print("\nТоп 15 категорий:")
    print(cat_item.head(15).to_string())

    fig, ax = plt.subplots(figsize=(10, 5))
    cat_item.head(20).plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
    ax.set_title("Топ-20 категорий по количеству объявлений")
    ax.set_xlabel("item_category_id")
    ax.set_ylabel("Количество объявлений")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.savefig("eda_category_items.png", dpi=120)
    plt.close()

Уникальных категорий в benchmark_items: 47
Объявлений: min=1, mean=4026, max=187336

Топ 15 категорий:
item_category_id
114    187336
19        357
112       141
40        122
33        103
10         98
20         84
25         82
36         74
111        74
39         72
24         70
83         68
27         57
30         56


In [ ]:
# Распределение запросов по категориям
if "search_category" in benchmark_queries.columns:
    cat_q = benchmark_queries["search_category"].value_counts()
    print(f"\nУникальных категорий в benchmark_queries: {len(cat_q)}")
    print("Топ 15:")
    print(cat_q.head(15).to_string())


Уникальных категорий в benchmark_queries: 2
Топ 15:
search_category
114    2230
0       222


In [ ]:
# Ключевая проверка: насколько хорошо категория запроса совпадает с категорией объявлений?
# Для этого смотрим на train: search_category vs item_category_id

if "search_category" in train.columns and "item_category_id" in train.columns:
    same_cat = (train["search_category"].astype(str) == train["item_category_id"].astype(str))
    print(f"\n=== СОВПАДЕНИЕ КАТЕГОРИЙ В TRAIN ===")
    print(f"Доля строк, где search_category == item_category_id: {same_cat.mean()*100:.1f}%")
    print()
    print("Вывод: если >90%, категория - жесткий фильтр при поиске,")
    print("       и можно смело ограничивать кандидатов только одной категорией.")


=== СОВПАДЕНИЕ КАТЕГОРИЙ В TRAIN ===
Доля строк, где search_category == item_category_id: 100.0%

Вывод: если >90%, категория - жесткий фильтр при поиске,
       и можно смело ограничивать кандидатов только одной категорией.


## 4. Пересечение запросов train / benchmark

**Ключевой вопрос**: сколько benchmark-запросов мы уже видели в train?
Для них мы знаем правильные объявления напрямую - это фактически бесплатный Recall.

In [ ]:
train_q_lower = train["search_query"].fillna("").str.lower().str.strip()
bench_q_lower = benchmark_queries["search_query"].fillna("").str.lower().str.strip()

train_q_set = set(train_q_lower)
bench_q_set = set(bench_q_lower)
overlap     = bench_q_set & train_q_set

print(f"=== ПЕРЕСЕЧЕНИЕ ЗАПРОСОВ ===")
print(f"Уникальных текстов запросов в train:     {len(train_q_set):,}")
print(f"Уникальных текстов запросов в benchmark: {len(bench_q_set):,}")
print(f"Совпадающих:                             {len(overlap):,}")
print(f"Покрытие benchmark через train lookup:   {len(overlap)/len(bench_q_set)*100:.1f}%")

print()
print("Вывод: для запросов, которые есть в train, можно напрямую возвращать")
print("       объявления с наибольшим числом кликов - без какого-либо обучения.")

=== ПЕРЕСЕЧЕНИЕ ЗАПРОСОВ ===
Уникальных текстов запросов в train:     74,529
Уникальных текстов запросов в benchmark: 2,452
Совпадающих:                             907
Покрытие benchmark через train lookup:   37.0%

Вывод: для запросов, которые есть в train, можно напрямую возвращать
       объявления с наибольшим числом кликов - без какого-либо обучения.


In [ ]:
# Сколько item_id из train присутствуют в benchmark_items?
VALID_ITEM_IDS = set(benchmark_items["item_id"])
train_valid_items_pct = train["item_id"].isin(VALID_ITEM_IDS).mean() * 100
print(f"\nДоля item_id из train, которые есть в benchmark_items: {train_valid_items_pct:.1f}%")
print("Именно эти объявления могут попасть в ответ через train lookup.")


Доля item_id из train, которые есть в benchmark_items: 6.6%
Именно эти объявления могут попасть в ответ через train lookup.


## 5. Статистика кликов в train

Понимание "плотности" релевантности помогает настроить N_CANDIDATES

In [ ]:
clicks_per_query = train.groupby("search_query")["item_id"].nunique()

print("=== КЛИКОВ НА ЗАПРОС (уникальных объявлений) ===")
print(clicks_per_query.describe().round(2).to_string())

fig, ax = plt.subplots(figsize=(9, 4))
counts_clipped = clicks_per_query.clip(upper=30)
ax.hist(counts_clipped, bins=30, color="teal", edgecolor="white")
ax.set_title("Распределение числа кликнутых объявлений на запрос (обрезано на 30)")
ax.set_xlabel("Кол-во уникальных объявлений")
ax.set_ylabel("Кол-во запросов")
plt.tight_layout()
plt.savefig("eda_clicks_per_query.png", dpi=120)
plt.close()

print()
print("Если медиана << 50, большинство запросов имеют немного релевантных объявлений,")
print("и Recall@50 с хорошим ретривером теоретически близок к 1 для многих запросов.")

=== КЛИКОВ НА ЗАПРОС (уникальных объявлений) ===
count    74529.00
mean         5.97
std         50.30
min          1.00
25%          1.00
50%          1.00
75%          3.00
max       5474.00

Если медиана << 50, большинство запросов имеют немного релевантных объявлений,
и Recall@50 с хорошим ретривером теоретически близок к 1 для многих запросов.


In [ ]:
# Запросы с очень большим числом релевантных объявлений - это "хвосты"
top_q = clicks_per_query.nlargest(10)
print("\nТоп-10 запросов по числу кликнутых объявлений:")
print(top_q.to_string())


Топ-10 запросов по числу кликнутых объявлений:
search_query
маникюр                    5474
наращивание ресниц         4189
массаж                     3830
педикюр                    2867
установка кондиционеров    2718
сантехник                  2576
электрик                   2451
вывоз мусора               2349
эвакуатор                  2236
покос травы                1961


## 6. Анализ текстов: заголовки и описания

In [ ]:
# Длина заголовков
if "item_title_raw" in benchmark_items.columns:
    title_len = benchmark_items["item_title_raw"].fillna("").str.len()
    print("=== ДЛИНА ЗАГОЛОВКОВ (символы) ===")
    print(title_len.describe().round(1).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(title_len.clip(upper=200), bins=40, color="steelblue", edgecolor="white")
    axes[0].set_title("Длина заголовка объявления")
    axes[0].set_xlabel("Символы")

    # Длина поисковых запросов
    if "search_query" in benchmark_queries.columns:
        qlen = benchmark_queries["search_query"].fillna("").str.len()
        axes[1].hist(qlen.clip(upper=100), bins=30, color="coral", edgecolor="white")
        axes[1].set_title("Длина поискового запроса")
        axes[1].set_xlabel("Символы")
        print(f"\nДлина поисковых запросов (benchmark): mean={qlen.mean():.1f}, max={qlen.max()}")

    plt.tight_layout()
    plt.savefig("eda_text_lengths.png", dpi=120)
    plt.close()

=== ДЛИНА ЗАГОЛОВКОВ (символы) ===
count    189212.0
mean         33.6
std          12.2
min           1.0
25%          24.0
50%          35.0
75%          44.0
max         100.0

Длина поисковых запросов (benchmark): mean=23.0, max=70


In [ ]:
# Заполненность infm_params - важно для BM25-фильтрации
if "item_infm_params_text" in benchmark_items.columns:
    has_infm_item = benchmark_items["item_infm_params_text"].notna() & \
                    (benchmark_items["item_infm_params_text"].str.len() > 0)
    print(f"\nДоля объявлений с заполненным item_infm_params_text: {has_infm_item.mean()*100:.1f}%")

if "search_infm_params_text" in benchmark_queries.columns:
    has_infm_q = benchmark_queries["search_infm_params_text"].notna() & \
                 (benchmark_queries["search_infm_params_text"].str.len() > 0)
    print(f"Доля запросов с заполненным search_infm_params_text: {has_infm_q.mean()*100:.1f}%")

    # Примеры параметров поиска
    sample_infm = benchmark_queries[has_infm_q]["search_infm_params_text"].head(5)
    print("\nПримеры search_infm_params_text:")
    for v in sample_infm:
        print(f"  {v}")


Доля объявлений с заполненным item_infm_params_text: 100.0%
Доля запросов с заполненным search_infm_params_text: 36.9%

Примеры search_infm_params_text:
  Вид услуги Деловые услуги
  Вид услуги Красота, здоровье
  Тип услуги Дизайн, рисование Вид услуги Обучение, курсы
  Вид услуги Деловые услуги Тип услуги Бухгалтерия, финансы
  Вид услуги Оборудование, производство


In [ ]:
# Какие слова чаще всего встречаются в заголовках?
if "item_title_raw" in benchmark_items.columns:
    all_words = []
    for t in benchmark_items["item_title_raw"].dropna().sample(min(20000, len(benchmark_items))):
        tokens = re.sub(r"[^а-яa-z\s]", " ", str(t).lower()).split()
        all_words.extend([w for w in tokens if len(w) >= 3])

    top_words = Counter(all_words).most_common(20)
    print("\n=== ТОП-20 СЛОВ В ЗАГОЛОВКАХ ===")
    for word, cnt in top_words:
        print(f"  {word:<20} {cnt:>6,}")


=== ТОП-20 СЛОВ В ЗАГОЛОВКАХ ===
  ремонт                2,061
  аренда                1,154
  услуги                  894
  под                     870
  репетитор               853
  для                     833
  ключ                    713
  работы                  550
  авто                    471
  установка               466
  заказ                   451
  грузоперевозки          430
  онлайн                  398
  вывоз                   381
  наращивание             379
  мастер                  366
  монтаж                  357
  замена                  350
  маникюр                 326
  обучение                301


## 7. Локации

In [ ]:
if "item_location_id" in benchmark_items.columns:
    loc_item = benchmark_items["item_location_id"].value_counts()
    print(f"=== ЛОКАЦИИ ===")
    print(f"Уникальных локаций в объявлениях: {len(loc_item)}")
    print(f"Топ локация занимает {loc_item.iloc[0] / len(benchmark_items) * 100:.1f}% объявлений")
    print("\nТоп 10 локаций по объявлениям:")
    print(loc_item.head(10).to_string())

if "search_location_id" in benchmark_queries.columns:
    loc_q = benchmark_queries["search_location_id"].value_counts()
    print(f"\nУникальных локаций в запросах: {len(loc_q)}")
    print("Топ 10 локаций по запросам:")
    print(loc_q.head(10).to_string())

=== ЛОКАЦИИ ===
Уникальных локаций в объявлениях: 2877
Топ локация занимает 10.3% объявлений

Топ 10 локаций по объявлениям:
item_location_id
637640    19543
653240    12145
633540     4494
650400     3659
641780     3178
654070     3157
640860     3070
652000     3027
661420     2726
625810     2575

Уникальных локаций в запросах: 273
Топ 10 локаций по запросам:
search_location_id
637640    329
653240    214
107620    119
621540     94
650400     75
633540     66
640860     50
107621     50
661420     49
652000     46


In [ ]:
# Проверяем: насколько локация запроса коррелирует с локацией объявления в train?
if "search_location_id" in train.columns and "item_location_id" in train.columns:
    same_loc = (train["search_location_id"].astype(str) == train["item_location_id"].astype(str))
    print(f"\nДоля строк в train, где search_location == item_location: {same_loc.mean()*100:.1f}%")
    print("Вывод: локация - сильный, но не обязательный фильтр;")
    print("       лучше использовать как бонус, а не жесткое ограничение.")


Доля строк в train, где search_location == item_location: 83.1%
Вывод: локация - сильный, но не обязательный фильтр;
       лучше использовать как бонус, а не жесткое ограничение.


## 8. Рейтинг и популярность

In [ ]:
if "item_rating" in benchmark_items.columns:
    r = benchmark_items["item_rating"].dropna()
    print("=== РЕЙТИНГИ ===")
    print(r.describe().round(3).to_string())

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(r, bins=25, color="gold", edgecolor="white")
    ax.set_title("Распределение рейтинга объявлений")
    ax.set_xlabel("Рейтинг")
    plt.tight_layout()
    plt.savefig("eda_ratings.png", dpi=120)
    plt.close()

if "item_rating_reviews_count" in benchmark_items.columns:
    rc = benchmark_items["item_rating_reviews_count"].dropna()
    print(f"\n=== ОТЗЫВЫ ===")
    print(rc.describe().round(1).to_string())
    print(f"\nМедиана отзывов: {rc.median():.0f} - большинство объявлений имеют мало отзывов.")
    print("log(1+reviews) * rating - хорошая мера популярности для tie-breaking.")

=== РЕЙТИНГИ ===
count    171581.000
mean          4.789
std           0.736
min           0.000
25%           4.876
50%           5.000
75%           5.000
max           5.000

=== ОТЗЫВЫ ===
count    177597.0
mean         66.5
std         230.8
min           0.0
25%           5.0
50%          18.0
75%          53.0
max       18291.0

Медиана отзывов: 18 - большинство объявлений имеют мало отзывов.
log(1+reviews) * rating - хорошая мера популярности для tie-breaking.


## 9. Доставка и специальные флаги

In [ ]:
if "search_is_delivery_search" in benchmark_queries.columns:
    delivery_pct = benchmark_queries["search_is_delivery_search"].fillna(0).mean() * 100
    print(f"Доля запросов с поиском по доставке: {delivery_pct:.1f}%")

if "item_is_phone_hidden" in benchmark_items.columns:
    phone_hidden_pct = benchmark_items["item_is_phone_hidden"].fillna(0).mean() * 100
    print(f"Доля объявлений со скрытым телефоном: {phone_hidden_pct:.1f}%")

Доля запросов с поиском по доставке: 0.0%
Доля объявлений со скрытым телефоном: 15.6%


## 10. Анализ запросов, для которых нет совпадения в train

In [ ]:
# Запросы из benchmark, которых нет в train - для них нельзя использовать лукап
bench_no_train = bench_q_set - train_q_set
print(f"Benchmark-запросов, которых нет в train: {len(bench_no_train):,}")
print(f"Для {len(bench_no_train)/len(bench_q_set)*100:.1f}% запросов нужен BM25 или Dense Retrieval")
print()

# Примеры новых запросов
sample_new = list(bench_no_train)[:10]
print("Примеры 'новых' запросов (нет в train):")
for q in sample_new:
    row = benchmark_queries[benchmark_queries["search_query"].str.lower().str.strip() == q].iloc[0]
    cat = row.get("search_category", "N/A")
    print(f"  '{q}' (категория: {cat})")

Benchmark-запросов, которых нет в train: 1,545
Для 63.0% запросов нужен BM25 или Dense Retrieval

Примеры 'новых' запросов (нет в train):
  'тонары' (категория: 114)
  'баня  на калесах' (категория: 114)
  'орск такси' (категория: 114)
  'монтаж стеновых панелей в ванне пвх' (категория: 114)
  'такси от казани до челнов' (категория: 114)
  'заменить экран на телефоне' (категория: 114)
  'выкуп шин бу дисков и колес' (категория: 114)
  'семейный бассейн' (категория: 114)
  'целительница' (категория: 114)
  'смоленск фура' (категория: 0)


## 11. Структура описания объявлений

In [ ]:
if "item_description_raw" in benchmark_items.columns:
    desc_len = benchmark_items["item_description_raw"].fillna("").str.len()
    has_desc = (desc_len > 0).mean() * 100
    print(f"Доля объявлений с описанием: {has_desc:.1f}%")
    print(f"Длина описания: mean={desc_len[desc_len > 0].mean():.0f}, "
          f"median={desc_len[desc_len > 0].median():.0f}, "
          f"max={desc_len.max()}")
    print()
    print("Вывод: описание длинное, поэтому в BM25 берем только первые 200-300 символов.")
    print("Начало описания обычно содержит самую важную информацию.")

    # Пример хорошего и пустого описания
    sample_desc = benchmark_items[desc_len > 100]["item_description_raw"].iloc[0]
    print(f"\nПример описания:\n  {sample_desc[:300]!r}")

Доля объявлений с описанием: 100.0%
Длина описания: mean=1405, median=1054, max=8494

Вывод: описание длинное, поэтому в BM25 берем только первые 200-300 символов.
Начало описания обычно содержит самую важную информацию.

Пример описания:
  'Выезд на дом в любое время, вплоть до 23:00\n\n● Диагностика устройств;\n\n● Чистка от пыли и замена термопасты;\n\n● Проверка работоспособности всех компонентов.\n\nРемонт аппаратной части:\n\n● Замена и ремонт материнской платы;\n\n● Ремонт и замена жесткого диска (HDD/SSD);\n\n● Установка и замена оперативной '


## 12. Итоговые выводы: стратегия решения

На основе EDA формулируем архитектуру пайплайна:

### Что обнаружили

1. **Категория - жесткий фильтр** (>X% совпадений в train).
   Кандидаты почти всегда принадлежат той же категории, что и запрос.
   Поэтому BM25 строим отдельно на каждую категорию.

2. **Train lookup дает бесплатный Recall** для ~X% запросов.
   Для них знаем правильные объявления напрямую.
   Это самый сильный компонент пайплайна.

3. **Запросы короткие, заголовки тоже** - BM25 хорошо работает
   именно в этом режиме (короткий запрос vs. короткий документ).

4. **Локация - мягкий сигнал**: высокое совпадение, но не 100%.
   Используем как бонус, не как фильтр.

5. **Рейтинг и отзывы** - вспомогательный tie-breaker,
   не основной сигнал (объявления без отзывов тоже релевантны).

6. **infm_params** - структурированные фильтры.
   Добавляем к тексту запроса и документа для BM25.

### Итоговая архитектура

```
Компонент 1 (02_sparse.py):  Train Lookup + BM25
Компонент 2 (03_dense.py):   Dense Retrieval (pre-trained -> fine-tuned)
Финал      (04_aggregation.py): Объединение через RRF, answer.csv
```

In [ ]:
print("EDA завершен. Все графики сохранены в текущую директорию.")
print("Следующий шаг: 02_sparse.py - построение BM25 + Train Lookup.")

EDA завершен. Все графики сохранены в текущую директорию.
Следующий шаг: 02_sparse.py - построение BM25 + Train Lookup.
